# 04 Generators: training, sampling and NaN restoration

**Purpose:** train the synthetic-data generators (CTGAN, TVAE), sample synthetic student records, and restore missing values. All fidelity and classifier evaluation happens in later notebooks.

**Data rule:** only `train.csv` and files derived from it are read. **The test set is never loaded in this notebook.**

**Experiments**

| Section | Variant | Training table | Output tag |
|---|---|---|---|
| 1-2 | Baseline, no target | `gen_input_random.csv` | `{ctgan,tvae}_random` |
| 3-4 | Target as generator column, unbalanced (the paper's setting) | `gen_input_random_target.csv` | `{ctgan,tvae}_random_target` |
| 5 | Same generators, balanced by conditional sampling | saved models from section 3 | `..._random_target_balanced_restored.csv` |
| 6 | `from_column` strategy, target variant, unbalanced | `gen_input_from_column_target.csv` | `{ctgan,tvae}_from_column_target` |

**Settings (paper):** 300 epochs, seed 42. These move into `configs/base.yaml` during the refactor.

**Re-running:** a generator is retrained only if its `.pkl` and `.csv` files are missing, or if `FORCE_RETRAIN = True`. Otherwise the saved files are reused. Retraining is not guaranteed to be bit-identical to earlier runs, so keep `FORCE_RETRAIN = False` unless a rebuild is intended.

**Documented deviations and library notes**
- `Engagement_Level` is a categorical generator column in sections 3 to 6. This follows the paper's conditional-sampling design and differs from the build instructions, which discourage target-derived generator inputs. The label comes from training rows only.
- CTGAN uses `log_frequency=False`. The default (`True`) flattened the synthetic class share to 0.543 / 0.457 against the real 0.825 / 0.175. That discarded run is kept as `ctgan_random_target_logfreq_true.*`.
- SDV 2.0.0 interface changes (dictionary `fit`, `load_synthesizer`, no `cuda` argument) are documented in `00_api_checks.ipynb`.

In [2]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sdv
import torch
from sdv.metadata import Metadata
from sdv.sampling import Condition
from sdv.single_table import CTGANSynthesizer, TVAESynthesizer
from sdv.utils import load_synthesizer

# ---------- Settings (to move into configs/base.yaml) ----------
FEATURES = ["nevents", "ndays_act", "ncontent", "nforum_posts"]
FLAGS = ["nevents_missing", "ndays_act_missing", "nforum_posts_missing"]
TARGET = "Engagement_Level"
TABLE = "students"
SEED = 42
EPOCHS = 300
N_TOTAL = 21_328                  # size of the real training set
FORCE_RETRAIN = False             # True rebuilds everything (about 20 min)

GENERATORS = {
    "ctgan": (CTGANSynthesizer, dict(verbose=True, log_frequency=False)),
    "tvae": (TVAESynthesizer, dict()),
}


# ---------- Helpers ----------
def find_project_root(start=None):
    """Walk upwards until a folder containing configs/base.yaml is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "configs" / "base.yaml").exists():
            return folder
    raise FileNotFoundError(f"Could not find configs/base.yaml above {start}")


def unwrap(result):
    """Return a DataFrame whether SDV gave us a DataFrame or {table: DataFrame}."""
    if isinstance(result, dict):
        return result[TABLE] if TABLE in result else next(iter(result.values()))
    return result


def set_seeds(seed: int = SEED) -> None:
    """Seed NumPy and PyTorch."""
    np.random.seed(seed)
    torch.manual_seed(seed)


def build_metadata(df: pd.DataFrame, categorical=()) -> Metadata:
    """Detect SDV metadata, then mark the given columns as categorical."""
    metadata = Metadata.detect_from_dataframes({TABLE: df})
    for col in categorical:
        metadata.update_column(column_name=col, table_name=TABLE, sdtype="categorical")
    return metadata


def train_or_load(tag, cls, kwargs, real, metadata):
    """Train a generator and sample len(real) rows, unless saved files exist.

    Returns (raw synthetic DataFrame, training minutes or None if reused).
    """
    csv, pkl = GEN_DIR / f"{tag}.csv", GEN_DIR / f"{tag}.pkl"
    if csv.exists() and pkl.exists() and not FORCE_RETRAIN:
        print(f"[reuse] {tag}: saved files found, not retrained")
        return pd.read_csv(csv), None

    set_seeds()
    synth = cls(metadata, epochs=EPOCHS, **kwargs)
    start = time.time()
    synth.fit({TABLE: real})
    minutes = (time.time() - start) / 60
    fake = unwrap(synth.sample(num_rows=len(real)))
    fake.to_csv(csv, index=False)
    synth.save(str(pkl))
    print(f"[trained] {tag}: {minutes:.1f} min, shape {fake.shape}")
    return fake, minutes


def restore_random(df: pd.DataFrame, miss_rates: pd.Series, seed: int = SEED) -> pd.DataFrame:
    """Hide feature cells at the learned rates. The target column is never masked."""
    rng = np.random.default_rng(seed)
    cols = [c for c in FEATURES + [TARGET] if c in df.columns]
    out = df[cols].copy()
    out[FEATURES] = out[FEATURES].astype(float)
    for col in FEATURES:
        out.loc[rng.random(len(out)) < miss_rates[col], col] = np.nan
    return out


def restore_from_flags(df: pd.DataFrame) -> pd.DataFrame:
    """Set a feature to NaN where its generated flag is 1, then drop the flags."""
    assert set(np.unique(df[FLAGS].to_numpy())) <= {0, 1}, "flags must be 0/1"
    out = df[FEATURES + [TARGET]].copy()
    out[FEATURES] = out[FEATURES].astype(float)
    for col in FEATURES:
        flag = f"{col}_missing"
        if flag in df.columns:
            out.loc[df[flag] == 1, col] = np.nan
    return out


def sample_balanced(synth, n_per_class: int, seed: int = SEED) -> pd.DataFrame:
    """Conditionally sample equal numbers of each class, then shuffle."""
    parts = []
    for label in (0, 1):
        set_seeds(seed)
        cond = Condition(column_values={TARGET: label}, num_rows=n_per_class)
        parts.append(unwrap(synth.sample_from_conditions(conditions=[cond])))
    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)


# ---------- Paths and shared inputs ----------
PROJECT_ROOT = find_project_root()
PROC = PROJECT_ROOT / "data" / "processed"
GEN_DIR = PROJECT_ROOT / "artifacts" / "generators"
GEN_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(PROC / "train.csv")
MISS_RATES = train[FEATURES].isna().mean()      # learned from real train only
run_minutes = {}

print("sdv:", sdv.__version__, "| torch:", torch.__version__)
print("Project root:", PROJECT_ROOT)
print("train:", train.shape, "| missing rates:", (MISS_RATES * 100).round(2).to_dict())

sdv: 2.0.0 | torch: 2.14.1+cpu
Project root: D:\student_engagement_twin
train: (21328, 5) | missing rates: {'nevents': 17.08, 'ndays_act': 10.53, 'ncontent': 0.0, 'nforum_posts': 68.11}


## 1. Baseline generators (no target column)

**Goal:** train CTGAN and TVAE on the four imputed features only. This is the spec-compliant baseline: the generators never see the label.

**Input:** `gen_input_random.csv` (real train, KNN-imputed, whole numbers). All four columns are numerical, so `log_frequency` has no categorical column to act on here. The earlier baseline run used the default, and the saved files are reused.

**Output:** `ctgan_random.{csv,pkl}`, `tvae_random.{csv,pkl}` (gap-free synthetic rows, unlabelled), `metadata_random.json`.

**Expected:** shape (21,328, 4), integer values, no NaNs. Earlier training times: CTGAN 5.8 min, TVAE 3.0 min.

In [3]:
real_base = pd.read_csv(PROC / "gen_input_random.csv")[FEATURES]
meta_base = build_metadata(real_base)
meta_base.save_to_json(str(GEN_DIR / "metadata_random.json"), mode="overwrite")

raw_base = {}
for name, (cls, kwargs) in GENERATORS.items():
    raw_base[name], run_minutes[f"{name}_random"] = train_or_load(
        f"{name}_random", cls, kwargs, real_base, meta_base
    )
    print(f"   {name}: shape {raw_base[name].shape} | NaNs {int(raw_base[name].isna().sum().sum())}")

[reuse] ctgan_random: saved files found, not retrained
   ctgan: shape (21328, 4) | NaNs 0
[reuse] tvae_random: saved files found, not retrained
   tvae: shape (21328, 4) | NaNs 0


## 2. Restore NaNs for the baseline (`random` strategy)

**Goal:** the generators output gap-free rows, but real data has missing values. Put NaNs back at the real training rates so the synthetic data can be compared with real train.

**Method:** for each feature, hide cells at random at the missing rate learned from **real train** (fixed seed 42). Both generators receive the same mask, because the seed, rates and row count are identical.

**Output:** `ctgan_random_restored.csv`, `tvae_random_restored.csv`.

**Expected:** restored rates within about 0.5 points of real train (17.08 / 10.53 / 0.00 / 68.11%), `ncontent` exactly 0.

In [4]:
rates = {"real_train": MISS_RATES * 100}
for name in GENERATORS:
    restored = restore_random(raw_base[name], MISS_RATES)
    restored.to_csv(GEN_DIR / f"{name}_random_restored.csv", index=False)
    rates[name] = restored[FEATURES].isna().mean() * 100

print(pd.DataFrame(rates).round(2))

              real_train  ctgan   tvae
nevents            17.08  17.12  17.12
ndays_act          10.53  10.48  10.48
ncontent            0.00   0.00   0.00
nforum_posts       68.11  68.48  68.48


## 3. Target variant: generators trained with `Engagement_Level` (unbalanced)

**Goal:** the paper's main setting. The label is a categorical generator column, so each synthetic row carries its own label and the saved models support conditional sampling in section 5.

**Input:** `gen_input_random_target.csv` (imputed features plus the training label, from real train only).

**Decisions**
- `Engagement_Level` is declared **categorical** in the metadata. Left as detected, SDV may treat 0/1 as a number.
- CTGAN uses `log_frequency=False`, so the unconditional class share follows the real one. TVAE has no such option.

**Output:** `ctgan_random_target.{csv,pkl}`, `tvae_random_target.{csv,pkl}`, `metadata_random_target.json`.

**Expected class share (earlier runs):** CTGAN 0.824 / 0.176, TVAE 0.8435 / 0.1565. Real train: 0.8246 / 0.1754.

In [5]:
real_target = pd.read_csv(PROC / "gen_input_random_target.csv")[FEATURES + [TARGET]]
meta_target = build_metadata(real_target, categorical=[TARGET])
meta_target.save_to_json(str(GEN_DIR / "metadata_random_target.json"), mode="overwrite")
print({c: v["sdtype"] for c, v in meta_target.to_dict()["tables"][TABLE]["columns"].items()})

raw_target = {}
for name, (cls, kwargs) in GENERATORS.items():
    raw_target[name], run_minutes[f"{name}_random_target"] = train_or_load(
        f"{name}_random_target", cls, kwargs, real_target, meta_target
    )
    share = raw_target[name][TARGET].value_counts(normalize=True).round(4).to_dict()
    print(f"   {name}: class share {share} | real {real_target[TARGET].value_counts(normalize=True).round(4).to_dict()}")

{'nevents': 'numerical', 'ndays_act': 'numerical', 'ncontent': 'numerical', 'nforum_posts': 'numerical', 'Engagement_Level': 'categorical'}
[reuse] ctgan_random_target: saved files found, not retrained
   ctgan: class share {0: 0.824, 1: 0.176} | real {0: 0.8246, 1: 0.1754}
[reuse] tvae_random_target: saved files found, not retrained
   tvae: class share {0: 0.8435, 1: 0.1565} | real {0: 0.8246, 1: 0.1754}


## 4. Restore NaNs for the target variant (`random` strategy)

**Goal:** the same restoration as section 2, applied to the target-variant output.

**Decision:** `Engagement_Level` is **never** masked. A missing label would make the row unusable.

**Checks (assertions):** zero NaNs in the label, and label values identical before and after restoration.



In [6]:
rates = {"real_train": MISS_RATES * 100}
for name in GENERATORS:
    restored = restore_random(raw_target[name], MISS_RATES)
    assert restored[TARGET].isna().sum() == 0, "label must never be masked"
    assert restored[TARGET].equals(raw_target[name][TARGET]), "label values changed"
    restored.to_csv(GEN_DIR / f"{name}_random_target_restored.csv", index=False)
    rates[name] = restored[FEATURES].isna().mean() * 100

print(pd.DataFrame(rates).round(2))

              real_train  ctgan   tvae
nevents            17.08  17.12  17.12
ndays_act          10.53  10.48  10.48
ncontent            0.00   0.00   0.00
nforum_posts       68.11  68.48  68.48


## 5. Balanced synthetic sets (conditional sampling)

**Goal:** build the class-balanced training sets for the paper's balanced experiment. Each set has 10,664 rows per class (21,328 in total, the real training size), against about 82 / 18 in the unbalanced sets.

**Method:** reload the saved section-3 generators with `sdv.utils.load_synthesizer`, request each class with `Condition`, concatenate, shuffle (seed 42), then restore NaNs as in section 4. No model is retrained, and no real data is used apart from the missing rates.

**Checks (assertions):** exact 10,664 / 10,664 class counts and no label NaNs.

**Output:** `ctgan_random_target_balanced_restored.csv`, `tvae_random_target_balanced_restored.csv`. Existing files are reused unless `FORCE_RETRAIN` is set.

In [7]:
N_PER_CLASS = N_TOTAL // 2
for name in GENERATORS:
    out_path = GEN_DIR / f"{name}_random_target_balanced_restored.csv"
    if out_path.exists() and not FORCE_RETRAIN:
        df = pd.read_csv(out_path)
        print(f"[reuse] {name} balanced: {df.shape}, classes {df[TARGET].value_counts().sort_index().to_dict()}")
        continue

    synth = load_synthesizer(str(GEN_DIR / f"{name}_random_target.pkl"))
    restored = restore_random(sample_balanced(synth, N_PER_CLASS), MISS_RATES)

    counts = restored[TARGET].value_counts().sort_index().to_dict()
    assert counts == {0: N_PER_CLASS, 1: N_PER_CLASS}, f"{name}: unbalanced {counts}"
    assert restored[TARGET].isna().sum() == 0, f"{name}: label contains NaN"
    restored.to_csv(out_path, index=False)
    print(f"[built] {name} balanced: {restored.shape}, classes {counts}")

[reuse] ctgan balanced: (21328, 5), classes {0: 10664, 1: 10664}
[reuse] tvae balanced: (21328, 5), classes {0: 10664, 1: 10664}


## 6. `from_column` strategy: generators trained with missingness flags (target variant, unbalanced)

**Purpose:** the paper's second missingness strategy. The generators learn which values are missing by modelling 0/1 flag columns, and the generated flags restore the NaNs. Unlike the `random` strategy, this can keep the link between missingness and behaviour (for example silent forum users).

**Input:** `gen_input_from_column_target.csv`: four imputed features, three flags (`nevents_missing`, `ndays_act_missing`, `nforum_posts_missing`) and the training label, all from real train. `ncontent` has no flag because it has no gaps.

**Method**
1. The three flags and the label are categorical in the metadata.
2. CTGAN (`log_frequency=False`) and TVAE are trained with the same settings as above.
3. Where a generated flag is 1, the matching feature becomes NaN, then the flag columns are dropped. The label is never masked.

**Checks printed**
- Flag rates and restored NaN rates, synthetic vs real train.
- **Class-1 share when each flag is 1 versus 0, real vs synthetic.** This tests whether the generators learned the link between missingness and engagement, which the `random` strategy cannot do.



**Run time:** about 10 minutes the first time (this is the only section that trains in a normal run).

In [8]:
real_fc = pd.read_csv(PROC / "gen_input_from_column_target.csv")[FEATURES + FLAGS + [TARGET]]
meta_fc = build_metadata(real_fc, categorical=FLAGS + [TARGET])
meta_fc.save_to_json(str(GEN_DIR / "metadata_from_column_target.json"), mode="overwrite")
print({c: v["sdtype"] for c, v in meta_fc.to_dict()["tables"][TABLE]["columns"].items()})


def class1_share_by_flag(df):
    """Share of class 1 among rows where each flag is 0 or 1."""
    return {f: df.groupby(f)[TARGET].mean().round(3).to_dict() for f in FLAGS}


flag_rates = {"real": real_fc[FLAGS].mean().round(4)}
nan_rates = {"real_train": MISS_RATES.round(4)}
by_flag = {"real": class1_share_by_flag(real_fc)}

for name, (cls, kwargs) in GENERATORS.items():
    fake, run_minutes[f"{name}_from_column_target"] = train_or_load(
        f"{name}_from_column_target", cls, kwargs, real_fc, meta_fc
    )
    restored = restore_from_flags(fake)
    assert restored[TARGET].isna().sum() == 0, "label must never be masked"
    restored.to_csv(GEN_DIR / f"{name}_from_column_target_restored.csv", index=False)

    flag_rates[name] = fake[FLAGS].mean().round(4)
    nan_rates[name] = restored[FEATURES].isna().mean().round(4)
    by_flag[name] = class1_share_by_flag(fake)
    print(f"   {name}: class share {fake[TARGET].value_counts(normalize=True).round(4).to_dict()}")

print("\nFlag rates (share of rows with flag = 1)")
print(pd.DataFrame(flag_rates))
print("\nRestored NaN rate per feature")
print(pd.DataFrame(nan_rates))
print("\nShare of class 1 by flag value {flag value: share}")
print(pd.DataFrame(by_flag))

{'nevents': 'numerical', 'ndays_act': 'numerical', 'ncontent': 'numerical', 'nforum_posts': 'numerical', 'nevents_missing': 'categorical', 'ndays_act_missing': 'categorical', 'nforum_posts_missing': 'categorical', 'Engagement_Level': 'categorical'}


Gen. (-00.94) | Discrim. (-00.05): 100%|██████████| 300/300 [08:50<00:00,  1.77s/it]


[trained] ctgan_from_column_target: 9.0 min, shape (21328, 8)
   ctgan: class share {0: 0.8352, 1: 0.1648}
[trained] tvae_from_column_target: 3.6 min, shape (21328, 8)
   tvae: class share {0: 0.8296, 1: 0.1704}

Flag rates (share of rows with flag = 1)
                        real   ctgan    tvae
nevents_missing       0.1708  0.2579  0.1509
ndays_act_missing     0.1053  0.0664  0.0037
nforum_posts_missing  0.6811  0.7083  0.6462

Restored NaN rate per feature
              real_train   ctgan    tvae
nevents           0.1708  0.2579  0.1509
ndays_act         0.1053  0.0664  0.0037
ncontent          0.0000  0.0000  0.0000
nforum_posts      0.6811  0.7083  0.6462

Share of class 1 by flag value {flag value: share}
                                      real                 ctgan  \
nevents_missing        {0: 0.201, 1: 0.05}   {0: 0.201, 1: 0.06}   
ndays_act_missing     {0: 0.185, 1: 0.091}  {0: 0.169, 1: 0.104}   
nforum_posts_missing  {0: 0.328, 1: 0.104}  {0: 0.287, 1: 0.114}   

     

## 7. Run log

**Purpose:** record how this notebook was run, for reproducibility. Written to `reports/generators_run_log.json`.

**Contents:** timestamp, library versions, seed, epochs, generator settings, training minutes (`null` where saved models were reused, since nothing was trained in that run), and the row count of every output file.

In [9]:
outputs = sorted(p.name for p in GEN_DIR.glob("*_restored.csv"))
log = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "versions": {"sdv": sdv.__version__, "torch": torch.__version__, "pandas": pd.__version__},
    "seed": SEED,
    "epochs": EPOCHS,
    "force_retrain": FORCE_RETRAIN,
    "generator_kwargs": {n: {k: v for k, v in kw.items()} for n, (_, kw) in GENERATORS.items()},
    "training_minutes_this_run": run_minutes,
    "restored_outputs": {
        name: int(len(pd.read_csv(GEN_DIR / name))) for name in outputs
    },
}
report_dir = PROJECT_ROOT / "reports"
report_dir.mkdir(exist_ok=True)
(report_dir / "generators_run_log.json").write_text(json.dumps(log, indent=2), encoding="utf-8")
print(json.dumps(log, indent=2))

{
  "timestamp_utc": "2026-10-25T16:36:49+00:00",
  "versions": {
    "sdv": "2.0.0",
    "torch": "2.14.1+cpu",
    "pandas": "2.3.3"
  },
  "seed": 42,
  "epochs": 300,
  "force_retrain": false,
  "generator_kwargs": {
    "ctgan": {
      "verbose": true,
      "log_frequency": false
    },
    "tvae": {}
  },
  "training_minutes_this_run": {
    "ctgan_random": null,
    "tvae_random": null,
    "ctgan_random_target": null,
    "tvae_random_target": null,
    "ctgan_from_column_target": 8.97506976922353,
    "tvae_from_column_target": 3.6474066217740377
  },
  "restored_outputs": {
    "ctgan_from_column_target_restored.csv": 21328,
    "ctgan_random_restored.csv": 21328,
    "ctgan_random_target_balanced_restored.csv": 21328,
    "ctgan_random_target_restored.csv": 21328,
    "tvae_from_column_target_restored.csv": 21328,
    "tvae_random_restored.csv": 21328,
    "tvae_random_target_balanced_restored.csv": 21328,
    "tvae_random_target_restored.csv": 21328
  }
}


## Results: `from_column` generators (target variant, unbalanced)

**Setup:** CTGAN (`log_frequency=False`) and TVAE, 300 epochs, seed 42, SDV 2.0.0, trained on `gen_input_from_column_target.csv` (four imputed features, three missingness flags and the training label, all from real train). Flags and label are categorical. Training time: CTGAN 9.0 min, TVAE 3.6 min (CPU).

**Measured**

| | Real train | CTGAN | TVAE |
|---|---|---|---|
| `nevents_missing` rate | 0.1708 | 0.2579 | 0.1509 |
| `ndays_act_missing` rate | 0.1053 | 0.0664 | 0.0037 |
| `nforum_posts_missing` rate | 0.6811 | 0.7083 | 0.6462 |
| Class share (0 / 1) | 0.8246 / 0.1754 | 0.8352 / 0.1648 | 0.8296 / 0.1704 |

Share of class 1 by flag value (flag 0 / flag 1):

| Flag | Real | CTGAN | TVAE |
|---|---|---|---|
| `nevents_missing` | 0.201 / 0.050 | 0.201 / 0.060 | 0.192 / 0.047 |
| `ndays_act_missing` | 0.185 / 0.091 | 0.169 / 0.104 | 0.171 / 0.000 |
| `nforum_posts_missing` | 0.328 / 0.104 | 0.287 / 0.114 | 0.249 / 0.128 |

**Interpretation**
- Flag rates are not reproduced as closely as in the `random` strategy (which was within 0.4 points). The largest gaps are CTGAN `nevents_missing` (+8.7 points) and TVAE `ndays_act_missing` (-10.2 points).
- TVAE nearly lost the `ndays_act_missing` category (about 79 of 21,328 rows have it set, against 2,246 in real train), so its restored data has almost no NaNs in `ndays_act`. Effect on the classifier: untested.
- Both generators reproduce the direction of the real link between missingness and engagement for every flag (missing values go with a lower share of class 1). CTGAN keeps more of the strength for `nforum_posts_missing` (class-1 ratio flag 0 / flag 1: real 3.2, CTGAN 2.5, TVAE 1.9, computed from the table above).
- TVAE's 0.000 for `ndays_act_missing` rests on about 79 rows and is not interpreted.
- Class shares are close to real for both generators.

**Limitations:** one seed per generator, so run-to-run variance is unknown; the flag-rate deviations may partly be seed noise.

